# ML for Healthcare & Predictive Analytics

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Harshini-Suresha/ml-healthcare-app/blob/main/ML_Healthcare_Demo.ipynb)

One notebook that walks through four supervised-learning projects end to end: **diabetes** (KNN), **breast cancer** (KNN vs Logistic Regression), **heart disease** (neural network) and **autism spectrum disorder** (three classifiers). Run the cells top to bottom. The diabetes and breast cancer sections run on bundled data. The heart section also runs on bundled data. The autism section asks for its CSV file.

> Educational demonstration built from public datasets. Not a medical device and not for clinical decisions.

| Project | Method | Data |
|---|---|---|
| Diabetes | KNN, preprocessing comparison, tuning k | 768 records, 8 features (`data/diabetes.csv`) |
| Breast cancer | KNN with CV-chosen K vs grid-searched Logistic Regression | Wisconsin diagnostic data from scikit-learn |
| Heart disease | 13-8-14-1 neural network | 303 records, 13 attributes + `target` (`heart.csv`) |
| Autism | Cleaning, oversampling, LR / boosted trees / SVM | Kaggle "Autism Prediction" `train.csv` |

In [ ]:
import os, sys, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, RepeatedStratifiedKFold, StratifiedKFold
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.utils import resample
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, confusion_matrix,
                             classification_report, roc_auc_score, roc_curve, ConfusionMatrixDisplay)

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.alpha": .25})
TEAL, RED, GREY = "#1d6b55", "#c8283b", "#8fa89a"
REPO_RAW = "https://raw.githubusercontent.com/Harshini-Suresha/ml-healthcare-app/main/data/"

def get_file(name):
    """Find a data file locally, download it from the repo, or (in Colab) ask for an upload."""
    for p in (f"data/{name}", name):
        if os.path.exists(p): return p
    try:
        import urllib.request
        urllib.request.urlretrieve(REPO_RAW + name, name); return name
    except Exception:
        pass
    if "google.colab" in sys.modules:
        from google.colab import files
        print(f"Please upload {name}"); up = files.upload()
        if up: os.rename(next(iter(up)), name); return name
    return None

def scores(y, pred, prob=None):
    out = {"Accuracy": accuracy_score(y, pred), "Precision": precision_score(y, pred, zero_division=0),
           "Recall": recall_score(y, pred, zero_division=0), "F1": f1_score(y, pred, zero_division=0)}
    if prob is not None: out["ROC-AUC"] = roc_auc_score(y, prob)
    return pd.Series(out).round(3)

## 1. Diabetes: K-Nearest Neighbors

768 patient records, 8 diagnostic features and a binary `Outcome`. In five columns a **zero means the value is missing**, so how we treat those zeros is one of the things we compare.

In [ ]:
df = pd.read_csv(get_file("diabetes.csv"))
FEATURES = list(df.columns[:-1]); ZERO_COLS = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
print(df.shape, "| diabetic:", int(df.Outcome.sum()), "| non-diabetic:", int((df.Outcome == 0).sum()))
print("\nZeros that mean missing:"); print((df[ZERO_COLS] == 0).sum().sort_values(ascending=False))
df.describe().T.round(2)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
for ax, f in zip(axes.ravel(), FEATURES):
    d = df[df[f] != 0] if f in ZERO_COLS else df
    edges = np.histogram_bin_edges(d[f], bins=20)
    for o, c, l in [(0, TEAL, "Non-diabetic"), (1, RED, "Diabetic")]:
        ax.hist(d.loc[d.Outcome == o, f], bins=edges, alpha=.65, color=c, label=l)
    ax.set_title(f, fontsize=10)
axes[0, 0].legend(frameon=False); plt.tight_layout(); plt.show()

c = df.corr()
plt.figure(figsize=(6.5, 5.5)); plt.imshow(c, cmap="RdYlGn", vmin=-1, vmax=1); plt.colorbar(shrink=.8)
plt.xticks(range(len(c)), c.columns, rotation=60, ha="right"); plt.yticks(range(len(c)), c.columns)
for i in range(len(c)):
    for j in range(len(c)): plt.text(j, i, f"{c.iloc[i, j]:.2f}", ha="center", va="center", fontsize=7)
plt.title("Correlation matrix"); plt.grid(False); plt.tight_layout(); plt.show()
print("Correlation with Outcome:"); print(c["Outcome"].drop("Outcome").sort_values(ascending=False).round(2))

### Preprocessing study

Same model (KNN, k = 5), nine preprocessing combinations: three ways to treat the zeros times three scalers. Imputation and scaling are fitted on the training split only.

In [ ]:
def prepare(df, zero_mode, scaler, test_size=0.3, seed=42):
    d = df[(df[ZERO_COLS] != 0).all(axis=1)] if zero_mode == "Drop rows" else df.copy()
    X, y = d[FEATURES].astype(float), d.Outcome
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=test_size, random_state=seed, stratify=y)
    Xtr, Xte = Xtr.copy(), Xte.copy()
    if zero_mode == "Median impute":
        med = Xtr[ZERO_COLS].replace(0, np.nan).median()
        for part in (Xtr, Xte): part[ZERO_COLS] = part[ZERO_COLS].replace(0, np.nan).fillna(med)
    sc = {"None": None, "StandardScaler": StandardScaler(), "MinMaxScaler": MinMaxScaler()}[scaler]
    if sc is not None: Xtr, Xte = sc.fit_transform(Xtr), sc.transform(Xte)
    return np.asarray(Xtr), np.asarray(Xte), ytr.values, yte.values

rows = []
for z in ["Keep zeros", "Median impute", "Drop rows"]:
    for s in ["None", "StandardScaler", "MinMaxScaler"]:
        Xtr, Xte, ytr, yte = prepare(df, z, s)
        m = KNeighborsClassifier(5).fit(Xtr, ytr)
        rows.append((z, s, m.score(Xtr, ytr), m.score(Xte, yte)))
study = pd.DataFrame(rows, columns=["Zeros", "Scaling", "Train", "Test"]).assign(Gap=lambda t: t.Train - t.Test)
study.sort_values("Test", ascending=False).round(3)

In [ ]:
Xtr, Xte, ytr, yte = prepare(df, "Median impute", "StandardScaler")
ks = range(1, 26); tr, te = [], []
for k in ks:
    m = KNeighborsClassifier(k).fit(Xtr, ytr); tr.append(m.score(Xtr, ytr)); te.append(m.score(Xte, yte))
plt.figure(figsize=(7, 3.6)); plt.plot(ks, tr, color=GREY, lw=2, label="Train"); plt.plot(ks, te, color=TEAL, lw=2, label="Test")
plt.xlabel("k"); plt.ylabel("Accuracy"); plt.legend(frameon=False); plt.title("Accuracy by k (median-imputed, standardized)"); plt.show()

best_k = int(ks[int(np.argmax(te))]); m = KNeighborsClassifier(best_k).fit(Xtr, ytr); pred = m.predict(Xte)
print(f"best k on the test split: {best_k}, test accuracy {m.score(Xte, yte):.3f}")
ConfusionMatrixDisplay.from_predictions(yte, pred, cmap="Greens", display_labels=["No", "Yes"]); plt.grid(False); plt.show()
print(classification_report(yte, pred, target_names=["Non-diabetic", "Diabetic"]))

## 2. Breast cancer: KNN vs tuned Logistic Regression

Uses the Wisconsin diagnostic dataset that ships with scikit-learn (569 records, 30 features). **K** for KNN is chosen by cross-validation. Logistic Regression is tuned with `GridSearchCV` over a `RepeatedStratifiedKFold`. Missing values (none in this copy) would be mean-imputed; features are standardized.

In [ ]:
from sklearn.datasets import load_breast_cancer
bc = load_breast_cancer(as_frame=True).frame.copy(); bc["malignant"] = 1 - bc.pop("target")
X, y = bc.drop(columns="malignant"), bc.malignant
print(bc.shape, "| malignant:", int(y.sum()), "| benign:", int((y == 0).sum()))
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=.25, random_state=42, stratify=y)
pipe = lambda model: Pipeline([("imp", SimpleImputer(strategy="mean")), ("sc", StandardScaler()), ("m", model)])

cv = StratifiedKFold(5, shuffle=True, random_state=42); ks = list(range(1, 26))
cvacc = [cross_val_score(pipe(KNeighborsClassifier(k)), Xtr, ytr, cv=cv).mean() for k in ks]
best_k = ks[int(np.argmax(cvacc))]
plt.figure(figsize=(7, 3.4)); plt.plot(ks, cvacc, color=TEAL, lw=2, marker="o", ms=3); plt.axvline(best_k, color=RED, ls="--")
plt.xlabel("K"); plt.ylabel("CV accuracy"); plt.title(f"Cross-validated accuracy by K (best K = {best_k})"); plt.show()

grid = GridSearchCV(pipe(LogisticRegression(max_iter=3000)), {"m__C": [.01, .1, 1, 10, 100], "m__solver": ["lbfgs", "liblinear"]},
                    cv=RepeatedStratifiedKFold(n_splits=5, n_repeats=2, random_state=42), scoring="accuracy").fit(Xtr, ytr)
print("Logistic Regression best params:", grid.best_params_, "| CV accuracy:", round(grid.best_score_, 3))

In [ ]:
models = {f"KNN (K={best_k})": pipe(KNeighborsClassifier(best_k)).fit(Xtr, ytr), "Logistic Regression": grid.best_estimator_}
table, plt_data = {}, []
for name, mdl in models.items():
    pred, prob = mdl.predict(Xte), mdl.predict_proba(Xte)[:, 1]
    table[name] = scores(yte, pred, prob); plt_data.append((name, *roc_curve(yte, prob)[:2], table[name]["ROC-AUC"]))
    print(name); print(confusion_matrix(yte, pred))
display(pd.DataFrame(table)) if "display" in globals() else print(pd.DataFrame(table))
plt.figure(figsize=(4.8, 4))
for (n, f, t, a), c in zip(plt_data, [TEAL, RED]): plt.plot(f, t, color=c, lw=2, label=f"{n} (AUC {a:.3f})")
plt.plot([0, 1], [0, 1], "--", color="#b3cdb1"); plt.legend(frameon=False, loc="lower right"); plt.xlabel("False positive rate"); plt.ylabel("True positive rate"); plt.show()

## 3. Heart disease: neural network

A 13-8-14-1 feed-forward network (ReLU, ReLU, sigmoid output; Adam; batch size 8; 100 epochs), standardized inputs, 70/30 split. The original notebook uses Keras; this version uses scikit-learn's `MLPClassifier` with the same layout and Keras-style small uniform starting weights so it runs anywhere.

**Data:** a CSV with the 13 attributes `age, sex, cp, trestbps, chol, fbs, restecg, thalach, exang, oldpeak, slope, ca, thal` and a `target` column. It is bundled as `data/heart.csv`.

In [ ]:
heart_path = get_file("heart.csv")
if heart_path is None:
    print("heart.csv not found: add it to the data/ folder to run this section.")
else:
    hd = pd.read_csv(heart_path); HF = ["age","sex","cp","trestbps","chol","fbs","restecg","thalach","exang","oldpeak","slope","ca","thal"]
    X, y = hd[HF].values.astype(float), hd["target"].values.astype(int)
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=.3, random_state=0)
    sc = StandardScaler(); Xtr, Xte = sc.fit_transform(Xtr), sc.transform(Xte)
    net = MLPClassifier(hidden_layer_sizes=(8, 14), activation="relu", solver="adam", batch_size=8, random_state=0); hist = []
    net.partial_fit(Xtr[:8], ytr[:8], classes=[0, 1]); rng = np.random.RandomState(0)  # Keras-style 'uniform' init
    net.coefs_ = [rng.uniform(-.05, .05, w.shape) for w in net.coefs_]; net.intercepts_ = [np.zeros_like(v) for v in net.intercepts_]
    for e in range(100):
        net.partial_fit(Xtr, ytr); hist.append((net.loss_, net.score(Xtr, ytr)))
    hist = np.array(hist); fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
    ax[0].plot(hist[:, 0], color=RED, lw=2); ax[0].set_title("Training loss"); ax[1].plot(hist[:, 1], color=TEAL, lw=2); ax[1].set_title("Training accuracy")
    for a in ax: a.set_xlabel("Epoch")
    plt.tight_layout(); plt.show()
    pred = net.predict(Xte); print(scores(yte, pred, net.predict_proba(Xte)[:, 1]))
    ConfusionMatrixDisplay.from_predictions(yte, pred, cmap="Greens"); plt.grid(False); plt.show()

## 4. Autism spectrum disorder: screening-based classification

Cleaning: `yes/no` to 1/0, unknown `?` and `others` merged into `Others`, two extreme negative `result` values removed. Features added: `sum_score` (total of the ten screening answers), `ageGroup`, `ind`; `age` is log-transformed. The training set is **randomly oversampled** to balance the classes, then standardized. Three models are compared with ROC-AUC computed from predicted labels.

**Data:** the Kaggle "Autism Prediction" `train.csv`. Save it as `data/autism_train.csv` (or upload it when asked).

In [ ]:
asd_path = get_file("autism_train.csv")
if asd_path is None:
    print("autism_train.csv not found: add it to the data/ folder to run this section.")
else:
    raw = pd.read_csv(asd_path)
    if "Class/ASD" not in raw.columns and "ASD" in raw.columns: raw = raw.rename(columns={"ASD": "Class/ASD"})
    A = [f"A{i}_Score" for i in range(1, 11)]
    d = raw.replace({"yes": 1, "no": 0, "?": "Others", "others": "Others"}); d = d[d["result"] > -5].copy()
    d["ageGroup"] = d["age"].apply(lambda a: "Toddler" if a < 4 else "Kid" if a < 12 else "Teenager" if a < 18 else "Young" if a < 40 else "Senior")
    for c in ["jaundice", "austim", "used_app_before"]: d[c] = d[c].astype(int)
    d["sum_score"] = d[A].sum(axis=1); d["ind"] = d["austim"] + d["used_app_before"] + d["jaundice"]; d["age"] = np.log(d["age"].astype(float))
    for c in d.columns:
        if not pd.api.types.is_numeric_dtype(d[c]): d[c] = LabelEncoder().fit_transform(d[c].astype(str))
    X = d.drop(columns=["ID", "age_desc", "used_app_before", "austim", "Class/ASD"]); y = d["Class/ASD"].astype(int)
    print(f"{len(raw)} raw records -> {len(d)} after cleaning | ASD share {y.mean():.1%}")
    pd.crosstab(d["sum_score"], d["Class/ASD"]).plot(kind="bar", color=[TEAL, RED], figsize=(7, 3.2), title="Cases by screening sum score"); plt.show()

In [ ]:
if asd_path is not None:
    Xtr, Xv, ytr, yv = train_test_split(X, y, test_size=.2, random_state=10)
    minority = int(np.argmin(np.bincount(ytr))); imin, imaj = np.where(ytr.values == minority)[0], np.where(ytr.values != minority)[0]
    extra = resample(imin, replace=True, n_samples=len(imaj) - len(imin), random_state=0); idx = np.concatenate([np.arange(len(ytr)), extra])
    Xo, yo = Xtr.values[idx], ytr.values[idx]; print("training rows:", len(ytr), "-> oversampled:", len(yo))
    sc = StandardScaler(); Xo, Xv_s = sc.fit_transform(Xo), sc.transform(Xv)
    try:
        from xgboost import XGBClassifier; boosted = ("XGBoost", XGBClassifier(eval_metric="logloss", random_state=0))
    except Exception:
        from sklearn.ensemble import HistGradientBoostingClassifier; boosted = ("Gradient Boosting", HistGradientBoostingClassifier(random_state=0))
    mods = {"Logistic Regression": LogisticRegression(max_iter=2000), boosted[0]: boosted[1], "SVM (RBF)": SVC(probability=True, random_state=0)}
    res = []
    for n, m in mods.items():
        m.fit(Xo, yo); res.append((n, roc_auc_score(yo, m.predict(Xo)), roc_auc_score(yv, m.predict(Xv_s))))
    res = pd.DataFrame(res, columns=["Model", "Train ROC-AUC", "Validation ROC-AUC"]).set_index("Model").round(3); print(res)
    res.plot(kind="bar", color=[GREY, TEAL], figsize=(7, 3.4), rot=0, ylim=(0, 1.08), title="Overfitting check: train vs validation"); plt.show()
    first = list(mods.values())[0]; ConfusionMatrixDisplay.from_estimator(first, Xv_s, yv, cmap="Greens"); plt.grid(False); plt.show()

## Notes

- Every number above is computed from the data when you run the notebook; nothing is hard-coded.
- The same logic powers the interactive Streamlit app in this repository (`app.py` and `ml_core.py`).
- Metrics are on a single train/test split unless a cross-validation is stated, so small differences between settings can be noise.
- Contact: [harshinisuresha7@gmail.com](mailto:harshinisuresha7@gmail.com)